## STW 任务原始数据处理

In [ ]:
import os
import sys
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
from tqdm import tqdm
from loguru import logger
from datetime import datetime, timedelta, timezone
from skyfield.api import load

logger.remove()
logger.add(sys.stderr, format="{time:YYYY-MM-DD HH:mm:ss} {level} {message}", level="INFO")

In [2]:
#! 设置全局变量
RAW_DATA_PATH = os.path.join('/home/dzp62442/Projects/CrossView_ws/data/')
LOC_A_PATH = os.path.join(RAW_DATA_PATH, 'Localization/Car-A')
LOC_B_PATH = os.path.join(RAW_DATA_PATH, 'Localization/Car-B')
PANO_PATH = os.path.join(RAW_DATA_PATH, 'SurroundView/VID_20231020_172334')
EXP_TIME = datetime(2023, 10, 20, 17) - timedelta(hours=8)  # 实验时间，由北京时间转为 UTC 时间
CAM_FPS = 29.97  # 摄像头帧率
CAM_SCENE_START_TIME = [1697793947132918, 1697794037132918, 1697794102132918, 1697794147132918]  # 各个相机序列的起始时间

### 1. 处理定位导航数据

In [23]:
# 组合导航的 GPS 时间戳转换为 Unix 时间戳
def gps_to_unix(gps_time):
    gps_start = datetime(1980, 1, 6)  # GPS时间起始于1980年1月6日
    gps_weeks = (EXP_TIME - gps_start).days // 7
    unix_time = gps_start + timedelta(weeks=gps_weeks, seconds=gps_time)
    unix_timestamp = unix_time.timestamp()
    return unix_timestamp

gps_to_unix(465854.41)

1697765054.41

In [24]:
# 使用 skyfield 库进行高精度的 GPS 时间戳转换
def gps_to_utc(gps_time):
    gps_start = datetime(1980, 1, 6)  # GPS时间起始于1980年1月6日
    gps_weeks = (EXP_TIME - gps_start).days // 7  # GPS周数
    ts = load.timescale()  # 加载天文历书和时间比例表
    t = ts.utc(1980, 1, 6) + timedelta(weeks=gps_weeks, seconds=gps_time)
    return t.utc_datetime().timestamp()

gps_to_utc(465854.41)

1697793836.41

In [25]:
# 将xlsx文件转换为csv文件
def xlsx_to_csv_pd(xlsx_file, csv_file):
    logger.info(f'xlsx_file: {xlsx_file}')
    df = pd.read_excel(xlsx_file, header=None)  # 不含表头
    columns_to_delete = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21]  # 从0开始索引
    df.drop(df.columns[columns_to_delete], axis=1, inplace=True)  # 删除指定的列
    tqdm.pandas(desc="Convert GPS time to UTC time")  # 显示进度条
    df.iloc[:, 0] = df.iloc[:, 0].progress_apply(gps_to_utc)  # 应用转换函数到指定列
    print(df.info())
    df.to_csv(csv_file, index=False, header=None)

In [26]:
# 处理定位导航数据
xlsx_file = os.path.join(LOC_A_PATH, 'A.xlsx')
csv_file = os.path.join(LOC_A_PATH, 'A.csv')
xlsx_to_csv_pd(xlsx_file, csv_file)
xlsx_file = os.path.join(LOC_B_PATH, 'B.xlsx')
csv_file = os.path.join(LOC_B_PATH, 'B.csv')
xlsx_to_csv_pd(xlsx_file, csv_file)

2023-11-23T19:36:01.487329+0800 INFO xlsx_file: /home/dzp62442/Projects/Cross-View/scripts/../raw_data/Localization/Car-A/A.xlsx


Convert GPS time to UTC time: 100%|██████████| 51417/51417 [01:11<00:00, 716.58it/s]
2023-11-23T19:37:18.483669+0800 INFO xlsx_file: /home/dzp62442/Projects/Cross-View/scripts/../raw_data/Localization/Car-B/B.xlsx


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51417 entries, 0 to 51416
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   10      51417 non-null  float64
 1   11      51417 non-null  float64
 2   12      51417 non-null  float64
 3   13      51417 non-null  float64
 4   14      51417 non-null  float64
 5   15      51417 non-null  float64
 6   16      51417 non-null  float64
 7   17      51417 non-null  float64
 8   18      51417 non-null  float64
 9   19      51417 non-null  float64
dtypes: float64(10)
memory usage: 3.9 MB
None


Convert GPS time to UTC time: 100%|██████████| 53097/53097 [01:13<00:00, 719.58it/s]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53097 entries, 0 to 53096
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   10      53097 non-null  float64
 1   11      53097 non-null  float64
 2   12      53097 non-null  float64
 3   13      53097 non-null  float64
 4   14      53097 non-null  float64
 5   15      53097 non-null  float64
 6   16      53097 non-null  float64
 7   17      53097 non-null  float64
 8   18      53097 non-null  float64
 9   19      53097 non-null  float64
dtypes: float64(10)
memory usage: 4.1 MB
None


### 2. 处理相机数据

In [ ]:
# 验证相机时间戳
camera_timestamp = 1697793947132918 / 1e6  # 转换为秒
camera_date_utc = datetime.utcfromtimestamp(camera_timestamp)  # UTC 时间
print("UTC time: ", camera_date_utc)
beijing_timezone = timezone(timedelta(hours=8))
camera_date_beijing = camera_date_utc.replace(tzinfo=timezone.utc).astimezone(beijing_timezone)
print("Beijing time: ", camera_date_beijing)

In [4]:
# 视频拆分为帧序列并生成时间戳文件
def process_videos(video_file, csv_file, fps, start_time):
    # 计算总帧数
    cap = cv.VideoCapture(video_file)
    if not cap.isOpened():
        logger.error(f'Fail to open video file: {video_file}')
        return 0
    else:
        logger.info(f'Open video file: {video_file}')
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    logger.info(f'Total frames: {total_frames}')
    
    # 生成帧编号和时间戳数据
    timestamps = [start_time]
    time_per_frame = 1 / fps
    for i in range(1, total_frames):
        timestamps.append(timestamps[-1] + time_per_frame)
    data = {
        "Frame": range(total_frames),
        "Timestamp": timestamps
    }
    df = pd.DataFrame(data)
    df.to_csv(csv_file, index=False, header=None)

    # 拆分视频为帧序列
    save_dir = os.path.join(os.path.dirname(video_file), 'imgs')
    os.makedirs(save_dir, exist_ok=True)
    for idx in tqdm(range(total_frames)):
        ret, frame = cap.read()
        if not ret:
            break  # 视频结束或读取错误
        frame_filename = os.path.join(save_dir, f"{idx}.jpg")
        cv.imwrite(frame_filename, frame)

    cap.release()


In [5]:
# 处理图像数据
for i, start_time in enumerate(CAM_SCENE_START_TIME):
    scene = 'scene' + str(i+1)
    video_path = os.path.join(PANO_PATH, scene, 'pano_clip'+str(i+1)+'.mp4')
    csv_file = os.path.join(PANO_PATH, scene, scene+'.csv')
    process_videos(video_path, csv_file, CAM_FPS, start_time/1e6)

2023-11-25 10:26:27 INFO Open video file: /home/dzp62442/Projects/CrossView_ws/data/SurroundView/VID_20231020_172334/scene1/pano_clip1.mp4
2023-11-25 10:26:27 INFO Total frames: 450
100%|██████████| 450/450 [00:32<00:00, 13.78it/s]
2023-11-25 10:26:59 INFO Open video file: /home/dzp62442/Projects/CrossView_ws/data/SurroundView/VID_20231020_172334/scene2/pano_clip2.mp4
2023-11-25 10:26:59 INFO Total frames: 450
100%|██████████| 450/450 [00:34<00:00, 13.19it/s]
2023-11-25 10:27:33 INFO Open video file: /home/dzp62442/Projects/CrossView_ws/data/SurroundView/VID_20231020_172334/scene3/pano_clip3.mp4
2023-11-25 10:27:33 INFO Total frames: 450
100%|██████████| 450/450 [00:35<00:00, 12.71it/s]
2023-11-25 10:28:09 INFO Open video file: /home/dzp62442/Projects/CrossView_ws/data/SurroundView/VID_20231020_172334/scene4/pano_clip4.mp4
2023-11-25 10:28:09 INFO Total frames: 450
100%|██████████| 450/450 [00:33<00:00, 13.35it/s]
